In [ ]:
# --- Act 0: Setup (hidden) ---
import warnings
warnings.filterwarnings('ignore')

import html
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from textwrap import shorten

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats
from IPython.display import HTML, display, Markdown

# --- Path resolution ---
_here = Path('.').resolve()
if (_here / 'projects').exists():
    REPO_ROOT = _here
elif (_here.parent / 'projects').exists():
    REPO_ROOT = _here.parent
else:
    raise FileNotFoundError(
        "Cannot find 'projects/' directory. "
        "Run this notebook from the repository root or the notebooks/ directory."
    )
CYCLE = REPO_ROOT / 'projects' / 'owasp-llm' / 'cycles' / '2026'

# --- Plotly config ---
pio.renderers.default = "notebook+png"
pio.templates.default = "plotly_white"

# --- Seaborn / matplotlib theme ---
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

# --- Color palette ---
ENTRY_IDS = [
    'LLM01', 'LLM02', 'LLM03', 'LLM04', 'LLM05',
    'LLM06', 'LLM07', 'LLM08', 'LLM09', 'LLM10',
    'NEW-ITSCD', 'NEW-MA', 'NEW-MSDA', 'NEW-MTIE', 'NEW-PMP', 'NEW-WLA',
    'ROLL-CFAS', 'ROLL-CMSB', 'ROLL-LAPTF', 'ROLL-SICG',
]
FRAME_BLIND = {'LLM04', 'LLM08', 'LLM10'}

_incumbent_blues = sns.color_palette('mako', n_colors=12)
_new_oranges = sns.color_palette('flare', n_colors=8)
_roll_purples = sns.color_palette('crest', n_colors=6)
ENTRY_COLORS = {}
_ib = 0; _in = 0; _ir = 0
for eid in ENTRY_IDS:
    if eid in FRAME_BLIND:
        ENTRY_COLORS[eid] = '#999999'
    elif eid.startswith('LLM'):
        ENTRY_COLORS[eid] = _incumbent_blues[_ib % len(_incumbent_blues)]
        _ib += 1
    elif eid.startswith('NEW'):
        ENTRY_COLORS[eid] = _new_oranges[_in % len(_new_oranges)]
        _in += 1
    elif eid.startswith('ROLL'):
        ENTRY_COLORS[eid] = _roll_purples[_ir % len(_roll_purples)]
        _ir += 1

# --- Deep-dive sidebar helper ---
def sidebar(title, content):
    """Render a collapsible deep-dive sidebar."""
    return HTML(
        f'<details style="margin: 1em 0; padding: 0.5em; '
        f'border-left: 3px solid #4a86c8; background: #f8f9fa;">'
        f'<summary style="cursor: pointer; font-weight: bold; '
        f'color: #2c5282;">{title}</summary>'
        f'<div style="margin-top: 0.8em; line-height: 1.6;">{content}</div>'
        f'</details>'
    )

def callout_box(text, border_color='#e53e3e', bg_color='#fff5f5'):
    """Render an always-visible boxed callout (not collapsed)."""
    return HTML(
        f'<div style="margin: 1em 0; padding: 1em; border-left: 4px solid {border_color}; '
        f'background: {bg_color}; line-height: 1.6;">{text}</div>'
    )

# --- Load all data ---
DATA = {}

with open(CYCLE / 'prereg' / 'rubric.json') as f:
    DATA['rubric'] = json.load(f)

with open(CYCLE / 'classify' / 'labeled_incidents_multimodel.json') as f:
    DATA['incidents'] = json.load(f)

prelabels = []
with open(CYCLE / 'calibration' / 'llm_prelabels.jsonl') as f:
    for line in f:
        prelabels.append(json.loads(line))
DATA['prelabels'] = prelabels

goldset = []
with open(CYCLE / 'calibration' / 'adjudicated_goldset.jsonl') as f:
    for line in f:
        goldset.append(json.loads(line))
DATA['goldset'] = goldset

precision_verif = []
with open(CYCLE / 'calibration' / 'precision_verification.jsonl') as f:
    for line in f:
        precision_verif.append(json.loads(line))
DATA['precision_verification'] = precision_verif

with open(CYCLE / 'calibration' / 'posteriors.json') as f:
    DATA['posteriors'] = json.load(f)

with open(CYCLE / 'calibration' / 'diagnostic.json') as f:
    DATA['diagnostic'] = json.load(f)

with open(CYCLE / 'infer' / 'inference_summary.json') as f:
    DATA['inference_summary'] = json.load(f)

DATA['lambda_samples'] = np.load(CYCLE / 'infer' / 'lambda_samples.npy')

with open(CYCLE / 'results' / 'concordance.json') as f:
    DATA['concordance'] = json.load(f)

with open(CYCLE / 'results' / 'selection_bias.json') as f:
    DATA['selection_bias'] = json.load(f)

with open(CYCLE / 'results' / 'rank_comparison_report.md') as f:
    DATA['rank_comparison_md'] = f.read()

# Build lookup tables
ENTRY_NAMES = {e['entry_id']: e['canonical_name'] for e in DATA['rubric']['entries']}
INFER_ENTRY_ORDER = DATA['inference_summary']['entry_ids']

# Validate shapes
assert DATA['lambda_samples'].shape == (16000, 20), (
    f"Expected lambda_samples shape (16000, 20), got {DATA['lambda_samples'].shape}"
)
assert len(INFER_ENTRY_ORDER) == 20, (
    f"Expected 20 entry_ids in inference_summary, got {len(INFER_ENTRY_ORDER)}"
)

print(f"Loaded {len(DATA)} data files. {len(DATA['incidents'])} incidents, "
      f"{len(DATA['prelabels'])} prelabels, {len(DATA['goldset'])} adjudications, "
      f"{DATA['lambda_samples'].shape[0]} posterior draws. Ready.")

# What the Data Says About the 2026 Top 10

## Act 1: The Question

The OWASP Top 10 for LLMs ranks AI security vulnerabilities. The 2025 list was built
from expert surveys — hundreds of security professionals voting on what matters most.
That process produced a consensus: Prompt Injection at #1, Sensitive Information
Disclosure at #2, and so on down to #10.

Expert opinion is one signal. We wanted to check it against a second signal:
the pattern of real-world incidents. We built a corpus of ~6,600 AI security incidents
from public databases, classified each one against the 20-entry taxonomy, and asked:
does the incident data agree with the experts?

This notebook walks through that analysis step by step. Along the way, you will see
how the classification worked, how we measured its accuracy, and what a Bayesian model
does with noisy measurements. Every chart and table below is computed live from the
data — you can re-run any cell to verify.

Here are the 20 taxonomy entries we are working with. The "Incident Rank" column is
blank for now. We will fill it in Act 6, after walking through the methodology.

In [ ]:
# Build the entry table with a placeholder rank column
entry_table = pd.DataFrame([
    {'#': i + 1, 'Entry ID': e['entry_id'], 'Name': e['canonical_name'], 'Incident Rank': '—'}
    for i, e in enumerate(DATA['rubric']['entries'])
])
entry_table = entry_table.set_index('#')
display(entry_table.style.set_caption(
    "20 taxonomy entries. Incident Rank will be filled in Act 6."
).set_properties(**{'text-align': 'left'}))

## Act 2: The Corpus

The corpus contains 6,639 incidents from public databases: CVE, GHSA, and OSV
(security advisories), plus AIAAIC (a database of AI-related harms and controversies).
Each record has a text description of what happened.

The corpus splits into two strata. The **security** stratum (CVE/GHSA/OSV) contains
things like prompt injection exploits, data leakage through APIs, and supply chain
compromises in ML packages. The **ai-harm** stratum (AIAAIC) contains things like
algorithmic discrimination, deepfake misuse, and surveillance overreach.

This split matters. The classifier performs differently on each stratum — security
incidents have more structured descriptions (CVE format), while ai-harm incidents
are written as news summaries with varying detail.

In [ ]:
# Count incidents by stratum
inc_df = pd.DataFrame(DATA['incidents'])
stratum_counts = inc_df['stratum'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart by stratum
colors = ['#2c5282', '#c05621']
stratum_counts.plot.barh(ax=axes[0], color=colors[:len(stratum_counts)])
axes[0].set_title('Incidents by Stratum')
axes[0].set_xlabel('Number of incidents')
for i, (stratum, count) in enumerate(stratum_counts.items()):
    axes[0].text(count + 30, i, f'{count:,}', va='center', fontsize=11)

# Bar chart by entry (top 15)
entry_counts = inc_df[inc_df['entry_id'] != 'out-of-scope']['entry_id'].value_counts().head(15)
bar_colors = [ENTRY_COLORS.get(eid, '#666666') for eid in entry_counts.index]
entry_counts.plot.barh(ax=axes[1], color=bar_colors)
axes[1].set_title('Top 15 Entries by Incident Count')
axes[1].set_xlabel('Number of incidents')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Show 2 real incident examples — one from each stratum
# Build stratum lookup from classified incidents (prelabels has no stratum field)
_stratum_lookup = {inc['incident_id']: inc['stratum'] for inc in DATA['incidents']}

prelabels_df = DATA['prelabels']
security_ex = next((p for p in prelabels_df if p['triage_tier'] == 'agree'
                    and p['consensus'] not in ('out-of-scope', None)
                    and _stratum_lookup.get(p['incident_id']) == 'security'), None)
harm_ex = next((p for p in prelabels_df if p['triage_tier'] == 'agree'
                and _stratum_lookup.get(p['incident_id']) == 'ai-harm'
                and len(p['text']) > 100), None)
if security_ex is None or harm_ex is None:
    display(HTML('<p style="color: red;">Could not find suitable example incidents.</p>'))

def incident_card(record, label):
    """Format an incident as an HTML card."""
    text = html.escape(shorten(record['text'], width=250, placeholder='...'))
    consensus = html.escape(str(record['consensus']))
    tier = html.escape(str(record['triage_tier']))
    inc_id = html.escape(str(record['incident_id']))
    return HTML(
        f'<div style="border: 1px solid #ccc; border-radius: 8px; padding: 1em; '
        f'margin: 0.5em 0; background: #fafafa;">'
        f'<strong>{html.escape(label)}</strong><br>'
        f'<code>{inc_id}</code> · consensus: '
        f'<strong>{consensus}</strong> · tier: {tier}<br>'
        f'<p style="margin-top: 0.5em; color: #333;">{text}</p>'
        f'</div>'
    )

display(HTML('<h4>Example: Security stratum (CVE/GHSA/OSV)</h4>'))
display(incident_card(security_ex, 'Security incident'))

display(HTML('<h4>Example: AI-harm stratum (AIAAIC)</h4>'))
display(incident_card(harm_ex, 'AI-harm incident'))

In [ ]:
# F-frame sidebar
display(sidebar(
    'Deep dive: What the corpus cannot see (F-frame)',
    '<p>The corpus is built from a keyword crawl of public databases — CVE, GHSA, OSV, '
    'and AIAAIC. Incidents that never became CVEs or harm-database entries are invisible '
    'to us. A prompt injection attack against an internal enterprise tool that was caught '
    'and patched quietly will never appear in this data.</p>'
    '<p>This creates structural bias toward vulnerability types that get reported in '
    'public channels. Well-resourced organizations that fix issues internally are '
    'underrepresented. Novel attack types that have not been assigned a CVE category '
    'are invisible.</p>'
))

# F-circ callout — always visible, not collapsed
display(callout_box(
    '<strong>Structural limitation: taxonomy-frame circularity (F-circ)</strong><br><br>'
    'We classified these incidents using the same taxonomy we are trying to validate. '
    'If the classifier systematically favors certain entries, the incident counts will '
    'appear to confirm the expert rankings even if the true pattern is different. '
    'This is taxonomy-frame circularity. It means the concordance we measure later '
    'is an upper bound on true agreement, not a precise estimate of it.',
    border_color='#d69e2e', bg_color='#fefcbf'
))

## Act 3: Classification — How We Labeled 6,600 Incidents

Each incident was classified by three different large language models: Qwen 235B,
Llama 405B, and DeepSeek V3. Each model independently read the incident text and
assigned it to one of the 20 taxonomy entries — or marked it "out of scope" if none
fit.

When all three models agreed on the same entry, we call it **agree tier**. When two
agreed and one differed, **split tier**. When all three picked different entries,
**disagree tier**.

The tier tells us how confident we can be in the classification. Agree-tier incidents
have strong consensus. Disagree-tier incidents sit in ambiguous territory where even
three independent classifiers could not converge.

In [ ]:
# Find a good agree-tier example with three matching votes
agree_ex = next((p for p in DATA['prelabels']
                 if p['triage_tier'] == 'agree'
                 and p['consensus'] not in ('out-of-scope', None)
                 and len(p['model_votes']) == 3
                 and len(p['text']) > 120), None)
assert agree_ex is not None, "No suitable agree-tier example found in prelabels"

display(HTML('<h4>Worked example: three-model classification</h4>'))
display(HTML(
    f'<div style="border: 1px solid #ccc; border-radius: 8px; padding: 1em; '
    f'margin: 0.5em 0; background: #f7fafc;">'
    f'<p style="color: #555;"><strong>Incident text</strong> (truncated):</p>'
    f'<p style="font-style: italic;">{html.escape(shorten(agree_ex["text"], 250, placeholder="..."))}</p>'
    f'<hr style="border: 0; border-top: 1px solid #e2e8f0;">'
    f'<table style="width: 100%; border-collapse: collapse;">'
    f'<tr style="background: #edf2f7;"><th>Model</th><th>Entry</th><th>Confidence</th></tr>'
    + ''.join(
        f'<tr><td>{html.escape(v["model_id"].split("/")[-1])}</td>'
        f'<td><strong>{html.escape(v["entry_id"])}</strong></td>'
        f'<td>{v["confidence"]:.0%}</td></tr>'
        for v in agree_ex['model_votes']
    )
    + f'</table>'
    f'<p style="margin-top: 0.5em;">Consensus: <strong>{html.escape(str(agree_ex["consensus"]))}</strong> '
    f'(tier: {html.escape(str(agree_ex["triage_tier"]))})</p>'
    f'</div>'
))

In [ ]:
# Tier distribution donut chart
tier_counts = Counter(p['triage_tier'] for p in DATA['prelabels'])
tier_labels = ['agree', 'split', 'disagree']
tier_values = [tier_counts.get(t, 0) for t in tier_labels]
tier_colors = ['#38a169', '#d69e2e', '#e53e3e']

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    tier_values, labels=None, autopct='%1.0f%%',
    colors=tier_colors, startangle=90, pctdistance=0.75,
    wedgeprops={'width': 0.4, 'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts:
    t.set_fontsize(13)
    t.set_fontweight('bold')

ax.legend(
    [f'{t}: {v:,}' for t, v in zip(tier_labels, tier_values)],
    loc='lower center', ncol=3, fontsize=11,
    bbox_to_anchor=(0.5, -0.05)
)
ax.set_title('Classification Tier Distribution\n(5,972 classified incidents)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Build entry-pair disagreement matrix from split and disagree tiers
in_scope_entries = [eid for eid in ENTRY_IDS]
confusion = pd.DataFrame(0, index=in_scope_entries, columns=in_scope_entries)

for p in DATA['prelabels']:
    if p['triage_tier'] in ('split', 'disagree'):
        votes = [v['entry_id'] for v in p['model_votes'] if v['entry_id'] != 'out-of-scope']
        unique_votes = list(set(votes))
        for i in range(len(unique_votes)):
            for j in range(i + 1, len(unique_votes)):
                a, b = unique_votes[i], unique_votes[j]
                if a in confusion.index and b in confusion.columns:
                    confusion.loc[a, b] += 1
                    confusion.loc[b, a] += 1

# Keep only entries with at least some disagreement
row_sums = confusion.sum(axis=1)
active = row_sums[row_sums > 5].index.tolist()
confusion_active = confusion.loc[active, active]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(confusion_active, dtype=bool), k=1)
sns.heatmap(
    confusion_active, mask=mask, cmap='YlOrRd', annot=True, fmt='d',
    linewidths=0.5, ax=ax, square=True, cbar_kws={'label': 'Disagreement count'}
)
ax.set_title('Entry-Pair Disagreements Across Split and Disagree Tiers', fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
display(sidebar(
    'Deep dive: Why three models?',
    '<p>Single-model classification had lower precision in our early experiments. '
    'A single model might confidently assign an incident to the wrong entry because '
    'of biases in its training data or the phrasing of the prompt.</p>'
    '<p>Three models with majority vote reduces noise the same way a panel of three '
    'judges reduces individual bias. If two of three models agree, we have higher '
    'confidence in the label. The disagree tier (where all three pick different entries) '
    'explicitly marks incidents where no classifier consensus exists.</p>'
))

display(sidebar(
    'Deep dive: The two-stage classification pipeline',
    '<p><strong>Stage 1 (heuristic):</strong> Regex and keyword indicators scan the '
    'incident text for known patterns (e.g., "prompt injection," "CVE-2026-*"). '
    'This produces a fast initial assignment at low confidence (10%). All incidents '
    'proceed to Stage 2 regardless of Stage 1 results.</p>'
    '<p><strong>Stage 2 (LLM):</strong> Each of three models reads the full incident '
    'text alongside the complete rubric (all 20 entries with inclusion/exclusion '
    'criteria). The model returns an entry_id, a confidence score, and a rationale. '
    'The three Stage 2 labels are combined into the consensus and triage tier.</p>'
))

## Act 4: How Good Is the Classifier?

The classifier is a tool, not ground truth. To trust the incident counts, we need
to measure how often the classifier gets it right — and how often it misses things.

**Precision**: When the classifier says "this incident belongs to LLM02," how often
is it correct? We verified 323 classifications by hand to measure this. Each entry
gets its own precision score — some entries are easier to classify than others.

**Recall**: Does the classifier find all incidents of a given type, or does it miss
some? A human reviewer adjudicated 1,200 incidents across all tiers to measure this.
The reviewer saw the incident text and the three model votes, then decided whether
to accept the consensus, override it, or mark the incident as out of scope.

**Stratum coverage note:** The 323 precision verifications were drawn entirely from
the security stratum (CVE/GHSA/OSV). The ai-harm stratum (AIAAIC) has no precision
measurements — the Bayesian model uses an empirical prior (the average across
measured entries) for ai-harm precision. This means error correction for ai-harm
incidents relies on borrowed estimates, not direct measurement.

In [ ]:
# Compute precision posterior means from posteriors.json
precision_data = DATA['posteriors']['precision']
diag = DATA['diagnostic']['entry_reports']

prec_rows = []
for key, params in precision_data.items():
    entry_id = key.split('::')[0]
    if entry_id == 'out-of-scope':
        continue
    alpha, beta = params['alpha'], params['beta']
    mean = alpha / (alpha + beta)
    n = int(alpha + beta - 2)  # subtract prior pseudocounts (1+1)
    prec_rows.append({
        'entry_id': entry_id,
        'precision_mean': mean,
        'sample_size': n,
        'alpha': alpha,
        'beta': beta,
    })

prec_df = pd.DataFrame(prec_rows).sort_values('precision_mean', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

# Compute 90% credible intervals
for i, row in enumerate(prec_df.itertuples()):
    ci_low, ci_high = stats.beta.ppf([0.05, 0.95], row.alpha, row.beta)
    color = ENTRY_COLORS.get(row.entry_id, '#666666')
    hatch = '//' if row.sample_size < 5 else ''

    ax.barh(i, row.precision_mean, color=color, alpha=0.7 if row.sample_size >= 5 else 0.35,
            edgecolor='#333', linewidth=0.5, hatch=hatch)
    ax.plot([ci_low, ci_high], [i, i], color='#333', linewidth=1.5, solid_capstyle='round')
    ax.text(ci_high + 0.02, i, f'n={row.sample_size}', va='center', fontsize=9, color='#555')

ax.set_yticks(range(len(prec_df)))
ax.set_yticklabels([f'{row.entry_id}' for row in prec_df.itertuples()], fontsize=10)
ax.set_xlabel('Precision (posterior mean with 90% CI)')
ax.set_title('Classifier Precision by Entry', fontsize=14)
ax.set_xlim(0, 1.15)
ax.axvline(0.5, color='#e53e3e', linestyle='--', alpha=0.5, label='50% precision')

# Add hatched-bar footnote
ax.text(0.02, -1.5, '// hatched bars = fewer than 5 precision observations (prior-dominated)',
        fontsize=9, color='#888', style='italic', transform=ax.get_yaxis_transform())

ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Show Beta posterior distributions for 4 key entries
key_entries = ['LLM03', 'LLM02', 'LLM06', 'LLM07']
x = np.linspace(0, 1, 500)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, eid in zip(axes.flat, key_entries):
    key = f'{eid}::security'
    if key not in precision_data:
        continue
    alpha = precision_data[key]['alpha']
    beta_param = precision_data[key]['beta']
    y = stats.beta.pdf(x, alpha, beta_param)
    mean = alpha / (alpha + beta_param)
    n = int(alpha + beta_param - 2)

    color = ENTRY_COLORS.get(eid, '#666666')
    ax.fill_between(x, y, alpha=0.3, color=color)
    ax.plot(x, y, color=color, linewidth=2)
    ax.axvline(mean, color=color, linestyle='--', alpha=0.7)
    ax.set_title(f'{eid}: {ENTRY_NAMES[eid]}\nmean={mean:.0%}, n={n}', fontsize=11)
    ax.set_xlabel('Precision')
    ax.set_ylabel('Density')
    ax.set_xlim(0, 1)

plt.suptitle('Precision Posterior Distributions (4 Key Entries)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
display(sidebar(
    'Deep dive: The gold-set process — 1,200 human adjudications',
    '<p>A human reviewer (the project author) adjudicated 1,200 incidents using a '
    'blind-first protocol. For each incident:</p>'
    '<ol>'
    '<li>Read the incident text without seeing the model votes (blind label).</li>'
    '<li>Record an independent classification.</li>'
    '<li>Then reveal the three model votes and the consensus.</li>'
    '<li>Make a final decision: accept the consensus, override to a different entry, '
    'assign multiple labels, or mark as out of scope.</li>'
    '</ol>'
    '<p>"Adjudication" is different from voting. The reviewer is not adding a fourth '
    'opinion — they are making a judgment call after seeing both the text and the '
    'model reasoning. The blind label (step 2) guards against anchoring to the '
    'model consensus.</p>'
))

# Full precision posteriors table
prec_table_rows = []
for key, params in sorted(precision_data.items()):
    entry_id = key.split('::')[0]
    if entry_id == 'out-of-scope':
        continue
    alpha, beta_p = params['alpha'], params['beta']
    mean = alpha / (alpha + beta_p)
    ci_low, ci_high = stats.beta.ppf([0.05, 0.95], alpha, beta_p)
    n = int(alpha + beta_p - 2)
    flag = '(prior-dominated)' if n < 5 else ''
    prec_table_rows.append({
        'Entry': entry_id,
        'alpha': alpha, 'beta': beta_p,
        'Mean': f'{mean:.1%}',
        '90% CI': f'[{ci_low:.1%}, {ci_high:.1%}]',
        'n': n,
        'Note': flag,
    })

prec_table_html = pd.DataFrame(prec_table_rows).to_html(index=False, escape=False)
display(sidebar('Deep dive: Full precision posteriors table', prec_table_html))

## Act 5: From Counts to Rankings — The Bayesian Model

Raw incident counts would be misleading. An entry whose classifier has 30% precision
looks like it has many incidents — but two-thirds of those are misclassifications
wrongly attributed to it.

We need a model that adjusts the observed counts for known classifier error. Think
of a bathroom scale that reads 2 pounds heavy. You would subtract 2 pounds from every
reading. The Bayesian model does this for each entry separately, and it carries the
uncertainty through — if the scale is 2±1 pounds off, the corrected weight is also
uncertain.

The model takes the observed incident counts, the measured precision and recall for
each entry, and produces a **posterior distribution** over the true incident rate for
each entry. A posterior distribution is not a single number — it is a range of
plausible values given the data. Wide distributions mean less certainty.

We drew 16,000 samples from this distribution using Markov chain Monte Carlo (MCMC).
MCMC is a method for sampling from probability distributions that are too complex to
compute directly. It generates a sequence of random samples that, after enough
iterations, represents the target distribution. Our run used 4 chains of 4,000
samples each, with 2,000 warmup iterations per chain.

**For 16 of 20 entries in the ai-harm stratum, we have not measured recall
directly.** The model uses a conservative prior estimate of ~1% recall for those
entries — Beta(1, 101). This means the model assumes the classifier finds very few
of those incidents and adjusts upward accordingly. These corrections are large, which
is one reason the credible intervals in Act 6 are wide.

In [ ]:
# Ridge / violin plot of posterior lambda for all 20 entries
lambda_samples = DATA['lambda_samples']
entry_order = INFER_ENTRY_ORDER

# Compute medians for sorting
medians = np.median(lambda_samples, axis=0)
sort_idx = np.argsort(medians)[::-1]

fig, ax = plt.subplots(figsize=(14, 10))

for plot_i, data_i in enumerate(sort_idx):
    eid = entry_order[data_i]
    samples = lambda_samples[:, data_i]
    is_frame_blind = eid in FRAME_BLIND

    # KDE for the ridge
    kde_x = np.linspace(samples.min(), np.percentile(samples, 99), 300)
    try:
        kde = stats.gaussian_kde(samples)
        kde_y = kde(kde_x)
    except Exception:
        continue

    # Normalize height
    kde_y = kde_y / kde_y.max() * 0.8

    color = '#999999' if is_frame_blind else ENTRY_COLORS.get(eid, '#4a86c8')
    alpha = 0.4 if is_frame_blind else 0.6

    ax.fill_between(kde_x, plot_i - kde_y, plot_i + kde_y,
                     alpha=alpha, color=color, linewidth=0)
    ax.plot(kde_x, plot_i + kde_y, color=color, linewidth=1)
    ax.plot(kde_x, plot_i - kde_y, color=color, linewidth=1)

    # Mark median
    med = medians[data_i]
    ax.plot(med, plot_i, 'o', color='white', markersize=5, zorder=5)
    ax.plot(med, plot_i, 'o', color=color, markersize=3, zorder=6)

    # Label
    label = f'{eid}'
    if is_frame_blind:
        label += ' (frame-blind) ★'
    ax.text(-0.001, plot_i, label, ha='right', va='center', fontsize=10,
            color='#999' if is_frame_blind else '#333',
            fontweight='normal' if is_frame_blind else 'bold')

ax.set_yticks([])
ax.set_xlabel('Posterior incident rate (λ)', fontsize=12)
ax.set_title('Posterior Distributions: True Incident Rate per Entry\n'
             '(★ = frame-blind: corpus cannot observe these entries)', fontsize=14)
ax.set_xlim(left=-0.0005)

plt.subplots_adjust(left=0.22)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics from lambda_samples and diagnostic.json
summary_rows = []
for i, eid in enumerate(INFER_ENTRY_ORDER):
    samples = lambda_samples[:, i]
    med = np.median(samples)
    ci_low, ci_high = np.percentile(samples, [5, 95])
    diag_report = DATA['diagnostic']['entry_reports'].get(eid, {})
    flag = diag_report.get('flag', '—')

    summary_rows.append({
        'Entry': eid,
        'Name': ENTRY_NAMES.get(eid, ''),
        '_median_numeric': med,
        'Median λ': f'{med:.4f}',
        '90% CI': f'[{ci_low:.4f}, {ci_high:.4f}]',
        'CI Width': f'{ci_high - ci_low:.4f}',
        'Diagnostic': flag,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('_median_numeric', ascending=False)
summary_df = summary_df.drop(columns=['_median_numeric'])
display(summary_df.style
    .set_caption('Posterior summary: median incident rate, 90% credible interval, diagnostic flag')
    .set_properties(**{'text-align': 'left'})
    .apply(lambda row: ['background: #f0f0f0' if row['Entry'] in FRAME_BLIND else ''
                        for _ in row], axis=1)
)

In [ ]:
# NumPyro model specification sidebar
model_source = (REPO_ROOT / 'engine' / 'model' / 'inference.py').read_text()
# Extract just the model function
model_start = model_source.find('def model(')
model_end = model_source.find('\n    # ------', model_start + 1)
if model_end == -1:
    model_end = model_source.find('\n    try:', model_start)
model_code = model_source[model_start:model_end]

display(sidebar(
    'Deep dive: The NumPyro model specification',
    '<p>This is the actual model we used, written in NumPyro (a probabilistic '
    'programming library for JAX). You do not need to install NumPyro to run this '
    'notebook — the results above are pre-computed.</p>'
    f'<pre style="background: #1a202c; color: #e2e8f0; padding: 1em; '
    f'border-radius: 4px; overflow-x: auto; font-size: 0.85em;">'
    f'{model_code.replace("<", "&lt;").replace(">", "&gt;")}</pre>'
    '<p><strong>Key components:</strong></p>'
    '<ul>'
    '<li><code>lambda</code>: latent prevalence per entry (HalfNormal prior)</li>'
    '<li><code>recall</code>, <code>precision</code>: per-entry, per-stratum '
    'measurement error (Beta priors from gold-set calibration)</li>'
    '<li><code>concentration</code>: over-dispersion parameter (Gamma prior)</li>'
    '<li>The FP leakage term (<code>einsum</code>) accounts for misclassified '
    'incidents that spill from one entry into another</li>'
    '<li>Negative-Binomial likelihood handles count over-dispersion</li>'
    '</ul>'
))

# MCMC diagnostics sidebar
inf_summary = DATA['inference_summary']
max_rhat = max(v for k, v in inf_summary['r_hat'].items() if k.startswith('lambda'))
min_ess = min(v for k, v in inf_summary['ess'].items() if k.startswith('lambda'))
total_draws = inf_summary['num_samples'] * inf_summary['num_chains']

display(sidebar(
    'Deep dive: MCMC convergence diagnostics',
    f'<p>The MCMC sampler ran <strong>{inf_summary["num_chains"]} chains</strong>, '
    f'each with <strong>{inf_summary["num_warmup"]:,}</strong> warmup iterations '
    f'and <strong>{inf_summary["num_samples"]:,}</strong> sampling iterations, '
    f'producing <strong>{total_draws:,}</strong> posterior draws total.</p>'
    '<p><strong>R-hat</strong> measures whether the chains converged to the same '
    'distribution. Values near 1.0 mean convergence. Our maximum R-hat across all '
    f'lambda parameters: <strong>{max_rhat:.6f}</strong> (threshold: ≤1.01). '
    'All parameters pass.</p>'
    f'<p><strong>Effective sample size (ESS)</strong> measures how many independent '
    f'samples the chains produced. Our minimum ESS for lambda parameters: '
    f'<strong>{min_ess:,.0f}</strong> out of {total_draws:,} draws '
    f'({min_ess/total_draws:.0%} efficiency). Higher is better.</p>'
    f'<p><strong>Divergences</strong>: <strong>{inf_summary["divergences"]}</strong>. '
    'Zero divergences means the sampler explored the posterior geometry without '
    'numerical problems. Any non-zero count would indicate regions the sampler '
    'could not traverse reliably.</p>'
))